# Guided Lab: Ingest & Store a Document Set — RouteMind

**Scenario:** Portside Logistics keeps five operations documents that RouteMind needs to search: the driver handbook, the safety and compliance guide, the customer SLA policy, hazmat handling procedures, and the vehicle maintenance manual. This lab loads all five as local PDFs, processes them using 3 chunking configurations, and embeds them using a local Hugging Face CPU model (falling back to a deterministic offline embedding if there's no network access).

This notebook follows the basic retrieval pipeline from the LangChain docs: load documents, split them into chunks, embed the chunks, and store them in a vector store. The official docs also note that Hugging Face embeddings can run locally on CPU and that local embeddings have lower latency than hosted APIs for short queries.

## Learning goals

By the end of this notebook, you should be able to:

1. Load 5 local PDF files into LangChain `Document` objects.
2. Add structured metadata to each document.
3. Split documents using 3 different chunking configurations.
4. Generate embeddings locally on CPU.
5. Store each chunk set in a local vector store.
6. Compare the chunk counts produced by each configuration.

## 1) Install packages

Run this once in a notebook cell:

```bash
pip install -U pypdf langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers reportlab
```

If you later want retriever traces, add `langsmith` as well.

In [ ]:
%pip install -qU pypdf langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers langsmith reportlab

## 2) Prepare your PDF folder

Place 5 local PDFs in a folder named `./pdfs`.

Example:

```text
pdfs/
├── driver_handbook.pdf
├── safety_compliance_guide.pdf
├── customer_sla_policy.pdf
├── hazmat_handling_procedures.pdf
└── vehicle_maintenance_manual.pdf
```

The LangChain PDF loader docs show that PDF files are one of the standard document ingestion sources. If you already have your own PDFs, drop them into `./pdfs` and skip ahead -- the next cell only generates sample documents when the folder doesn't already have 5 PDFs in it.

In [ ]:
# Generate Portside Logistics' 5 real operations documents if they aren't
# already present. This makes the lab runnable immediately with no manual
# file prep; replace any of these with your own PDFs at any time by placing
# a file with the same or a different name into ./pdfs.
from pathlib import Path

PDF_DIR = Path('./pdfs')
PDF_DIR.mkdir(exist_ok=True)
existing_pdfs = sorted(PDF_DIR.glob('*.pdf'))

if len(existing_pdfs) >= 5:
    print(f"Found {len(existing_pdfs)} PDFs already in {PDF_DIR}, skipping sample generation.")
else:
    print("Generating Portside Logistics' 5 sample operations documents...")

    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib.enums import TA_LEFT
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, ListFlowable, ListItem, Table, TableStyle
    )
    from reportlab.lib import colors

    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="DocTitle", fontSize=18, leading=22, spaceAfter=14, fontName="Helvetica-Bold"))
    styles.add(ParagraphStyle(name="SectionHead", fontSize=13, leading=16, spaceBefore=14, spaceAfter=6, fontName="Helvetica-Bold"))
    styles.add(ParagraphStyle(name="Body", fontSize=10.2, leading=14.5, spaceAfter=8, alignment=TA_LEFT))

    def build_pdf(filename, title, subtitle, sections):
        doc = SimpleDocTemplate(
            str(PDF_DIR / filename), pagesize=letter,
            topMargin=0.75 * inch, bottomMargin=0.75 * inch,
            leftMargin=0.85 * inch, rightMargin=0.85 * inch, title=title,
        )
        story = [Paragraph(title, styles["DocTitle"]), Paragraph(subtitle, styles["Body"]), Spacer(1, 10)]
        for heading, paragraphs, bullets in sections:
            story.append(Paragraph(heading, styles["SectionHead"]))
            for p in paragraphs:
                story.append(Paragraph(p, styles["Body"]))
            if bullets:
                story.append(ListFlowable(
                    [ListItem(Paragraph(b, styles["Body"])) for b in bullets],
                    bulletType="bullet", leftIndent=18,
                ))
        doc.build(story)
        print(f"  wrote {PDF_DIR / filename}")

    build_pdf(
        "driver_handbook.pdf", "Portside Logistics -- Driver Handbook",
        "Effective for all company and contract drivers operating out of the Chennai, Bengaluru, and Mumbai depots.",
        [
            ("1. Welcome and Purpose", [
                "This handbook sets the operating rules for every driver working a Portside Logistics route, "
                "whether you are a full-time company driver or a contracted owner-operator. It exists so that "
                "customers get a consistent experience no matter which depot dispatched the load, and so that "
                "every driver knows exactly what is expected before, during, and after a run.",
                "Dispatchers and depot managers use RouteMind, the internal operations assistant, to answer "
                "day-to-day questions about routes and shipments. This handbook is one of the source documents "
                "RouteMind is allowed to search when a driver or dispatcher asks a policy question.",
            ], None),
            ("2. Reporting for Duty", [
                "Drivers report to their home depot at the time shown on the weekly roster, not the time the "
                "shift technically starts. This gives time for the pre-shift briefing and the vehicle inspection "
                "described in the Safety and Compliance Guide.",
            ], [
                "Check in with the depot dispatcher and confirm your assigned vehicle and trailer number.",
                "Attend the pre-shift briefing, which covers route hazards, weather, and any customer notes.",
                "Complete the pre-trip inspection before leaving the yard.",
                "Log on to the driver app and confirm your first stop before departure.",
            ]),
            ("3. Hours of Service and Rest Breaks", [
                "No driver may exceed 10 hours of driving within a single shift. A mandatory 30-minute break is "
                "required after 5 continuous hours behind the wheel, and at least 8 hours of rest are required "
                "between the end of one shift and the start of the next.",
                "Drivers who anticipate exceeding these limits due to traffic, loading delays, or weather must "
                "radio dispatch before the limit is reached so the load can be reassigned or a relief driver "
                "arranged. Driving past the limit without dispatch approval is treated as a serious handbook "
                "violation regardless of the reason.",
            ], None),
            ("4. Route Deviations and Communication", [
                "Routes are planned by dispatch and should be followed as assigned. Any deviation expected to add "
                "more than 15 minutes to the plan -- a road closure, a customer request, or traffic -- must be "
                "called in to dispatch before the deviation, not after.",
                "Status updates (departed depot, arrived at stop, delivered, delayed) must be logged in the driver "
                "app in real time. This is the same status data that feeds the shipments table dispatchers and "
                "RouteMind query when a customer asks about a delivery.",
            ], None),
            ("5. Customer Interaction Standards", [
                "Drivers are the face of Portside Logistics at the customer's dock. Professional conduct is not "
                "optional: greet the receiving contact, confirm the shipment count matches the manifest, and "
                "obtain proof of delivery for every stop.",
            ], [
                "Capture a signature or a delivery photo at every stop; at least one is mandatory.",
                "If a delivery is refused, note the reason code in the app and contact dispatch before leaving.",
                "Never leave a shipment unattended at a location with no authorized receiver.",
                "Escalate any dispute about shipment condition to dispatch rather than negotiating on-site.",
            ]),
            ("6. Fuel Card and Expense Policy", [
                "Company fuel cards are for fuel, tolls, and approved parking only. Any other use must be "
                "pre-approved by the depot manager and supported by a receipt submitted within 5 business days.",
            ], None),
            ("7. Handbook Violations", [
                "Violations are logged by the depot manager and reviewed monthly. A first violation results in a "
                "documented warning; repeated violations within a rolling 12-month period can result in suspension "
                "of route assignments pending review.",
            ], None),
        ],
    )

    build_pdf(
        "safety_compliance_guide.pdf", "Portside Logistics -- Safety and Compliance Guide",
        "Applies to all depot staff, loading dock crew, and drivers across all Portside Logistics facilities.",
        [
            ("1. Purpose and Scope", [
                "This guide covers the minimum safety steps required before a vehicle leaves the yard, the "
                "protective equipment required on the dock and in the cab, and how to report an incident. It is "
                "reviewed every year alongside the Vehicle Maintenance Manual and the Hazmat Handling Procedures.",
            ], None),
            ("2. Pre-Trip Vehicle Inspection Checklist", [
                "Every driver completes a pre-trip inspection before the vehicle leaves the depot, and records the "
                "result in the maintenance log. A vehicle that fails any item below is held at the depot until a "
                "mechanic clears it.",
            ], [
                "Tire condition and pressure on all axles, including the spare.",
                "Headlights, tail lights, brake lights, and turn signals.",
                "Brake pedal feel and parking brake hold.",
                "Mirrors, windshield condition, and wiper function.",
                "Load securement points, straps, and tie-down rails.",
                "Engine oil, coolant, and washer fluid levels.",
            ]),
            ("3. Personal Protective Equipment", [
                "Steel-toe boots and a high-visibility vest are required on the loading dock at all times, no "
                "exceptions for depot visitors either. Gloves are required whenever handling strapped pallets or "
                "crates, and additional equipment is required for hazardous cargo as described in the Hazmat "
                "Handling Procedures.",
            ], None),
            ("4. Incident and Accident Reporting", [
                "Any accident, near-miss, or injury -- however minor -- must be reported to the depot safety "
                "officer within 1 hour of the event. Photograph the scene if it is safe to do so, and do not move "
                "a damaged vehicle until the safety officer says it is clear.",
            ], [
                "Call the depot safety officer immediately, then dispatch.",
                "Photograph the scene, the vehicle, and any damaged cargo.",
                "Complete the incident form before the end of the shift.",
                "For any injury, complete a medical follow-up within 24 hours regardless of severity.",
            ]),
            ("5. Load Securement Rules", [
                "Cargo must be secured so that it cannot shift more than a few centimeters in any direction under "
                "normal braking. Total load weight must stay within the rated capacity printed on the vehicle "
                "placard; overloaded vehicles will not be dispatched.",
            ], None),
            ("6. Adverse Weather Driving Protocol", [
                "During heavy rain or reduced visibility, drivers reduce speed and increase following distance, "
                "and dispatch may add buffer time to the route automatically. Hazmat runs are suspended entirely "
                "when wind speed exceeds the threshold set in the Hazmat Handling Procedures, and hill routes "
                "require chains or equivalent traction aids when the depot issues a weather advisory.",
            ], None),
        ],
    )

    build_pdf(
        "customer_sla_policy.pdf", "Portside Logistics -- Customer SLA Policy",
        "Governs delivery commitments, delay compensation, and claims for all Portside Logistics shipping customers.",
        [
            ("1. Service Tiers and Delivery Windows", [
                "Portside Logistics offers three service tiers. Standard service targets 3 to 5 business days "
                "between origin depot and destination city. Express service targets 1 to 2 business days on "
                "supported lanes. Same-Day service is available only for destinations within the same metro area "
                "as the origin depot and must be booked before the depot's daily cutoff time.",
            ], None),
            ("2. Delay Compensation Schedule", [
                "When a shipment is delivered later than the committed window, the customer is credited a "
                "percentage of the freight cost for that shipment automatically, without needing to file a claim.",
            ], [
                "Delayed by up to 4 hours past the committed window: no credit, courtesy notification only.",
                "Delayed by more than 4 and up to 24 hours: 10 percent of freight cost credited.",
                "Delayed by more than 24 and up to 48 hours: 25 percent of freight cost credited.",
                "Delayed by more than 48 hours: 50 percent of freight cost credited and priority rebooking.",
            ]),
            ("3. Damaged or Lost Shipment Claims", [
                "Claims for damaged or lost shipments must be filed within 72 hours of the delivery date, or of "
                "the committed delivery date if the shipment never arrived. A claim requires the tracking code, "
                "photographs of the damage where applicable, and the original packing list.",
                "Approved claims are paid out based on the declared value at booking, capped at the limit stated "
                "on the shipping contract for that customer tier.",
            ], None),
            ("4. Proof of Delivery Requirements", [
                "Every delivery requires a signature or a delivery photo captured by the driver at the time of "
                "drop-off. Proof of delivery records are retained for 180 days and can be requested by the "
                "customer at any time within that window through customer support.",
            ], None),
            ("5. Escalation Path for Unresolved Disputes", [
                "If a customer disagrees with a claims decision or a compensation credit, the first escalation "
                "point is the depot customer support lead. Unresolved disputes after that review move to the "
                "regional operations manager, who issues a final decision within 10 business days.",
            ], None),
        ],
    )

    build_pdf(
        "hazmat_handling_procedures.pdf", "Portside Logistics -- Hazmat Handling Procedures",
        "Mandatory for all staff and drivers involved in loading, transporting, or unloading hazardous cargo.",
        [
            ("1. Hazard Classes Carried by Portside Logistics", [
                "Portside Logistics is certified to carry a limited set of hazard classes: Class 3 flammable "
                "liquids, Class 8 corrosives, and Class 9 miscellaneous dangerous goods, each subject to the "
                "depot-specific volume limits set by the compliance team. Any shipment outside these classes must "
                "be declined at booking, not discovered at the dock.",
            ], None),
            ("2. Labeling and Placarding Requirements", [
                "Every hazmat package must carry the correct hazard class label before it reaches the loading "
                "dock, and the vehicle must display the matching placard for the highest hazard class on board "
                "before it leaves the depot. A vehicle missing the correct placard will not be dispatched.",
            ], None),
            ("3. Segregation and Loading Rules", [
                "Incompatible hazard classes may not be loaded in the same vehicle on the same run. Class 3 "
                "flammable liquids must be segregated from Class 8 corrosives by at least one empty bay or an "
                "approved segregation barrier, and loading staff confirm segregation before sign-off.",
            ], None),
            ("4. Personal Protective Equipment for Hazmat", [
                "Chemical-resistant gloves and splash goggles are required whenever handling Class 8 corrosives. "
                "A respirator rated for the specific chemical class is required for any Class 3 shipment with a "
                "damaged or leaking outer container, and that shipment must be isolated immediately rather than "
                "loaded.",
            ], None),
            ("5. Spill Response Steps", [
                "If a spill is discovered at the dock or on the road, the priority order is: stop the operation, "
                "secure the area so no one else approaches, and call the hazmat response line printed on the "
                "vehicle placard. Do not attempt cleanup beyond what the vehicle's spill kit is rated for, and "
                "never hose a spill toward a drain.",
            ], None),
            ("6. Driver Certification Requirement", [
                "Only drivers holding a current hazmat endorsement may be assigned a hazmat-carrying route. The "
                "endorsement must be renewed every 2 years, and dispatch will not assign a hazmat load to a driver "
                "whose endorsement expires within the trip window.",
            ], None),
        ],
    )

    service_table_data = [
        ["Service Item", "Interval"],
        ["Engine oil and filter change", "Every 10,000 km"],
        ["Brake inspection", "Every 20,000 km"],
        ["Tire rotation", "Every 15,000 km"],
        ["Full safety inspection", "Every 6 months"],
    ]
    doc5 = SimpleDocTemplate(
        str(PDF_DIR / "vehicle_maintenance_manual.pdf"), pagesize=letter,
        topMargin=0.75 * inch, bottomMargin=0.75 * inch,
        leftMargin=0.85 * inch, rightMargin=0.85 * inch,
        title="Portside Logistics -- Vehicle Maintenance Manual",
    )
    story = [
        Paragraph("Portside Logistics -- Vehicle Maintenance Manual", styles["DocTitle"]),
        Paragraph("Applies to all company-owned trucks and vans across the Chennai, Bengaluru, and Mumbai depots.", styles["Body"]),
        Spacer(1, 10),
        Paragraph("1. Purpose and Fleet Overview", styles["SectionHead"]),
        Paragraph(
            "This manual sets the maintenance schedule and out-of-service rules for the Portside Logistics fleet. "
            "Every service event, whether scheduled or a roadside repair, must be logged in the fleet maintenance "
            "system within 24 hours so the vehicle's service history stays accurate for the next inspection.",
            styles["Body"],
        ),
        Paragraph("2. Daily Checklist", styles["SectionHead"]),
        Paragraph(
            "Before any vehicle leaves the depot, the driver checks tire pressure on all axles, confirms all "
            "exterior lights are functioning, and looks underneath the vehicle for fresh fluid leaks. This is in "
            "addition to, not a replacement for, the full pre-trip inspection in the Safety and Compliance Guide.",
            styles["Body"],
        ),
        Paragraph("3. Scheduled Service Intervals", styles["SectionHead"]),
        Paragraph(
            "The table below lists the minimum service intervals for the standard fleet. Vehicles running hazmat "
            "routes follow a shorter brake and tire inspection interval set separately by the compliance team.",
            styles["Body"],
        ),
        Spacer(1, 6),
        Table(service_table_data, colWidths=[3.2 * inch, 2.2 * inch], style=TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2b3a55")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (-1, -1), 10),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f2f4f7")]),
            ("TOPPADDING", (0, 0), (-1, -1), 6),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ])),
        Spacer(1, 12),
        Paragraph("4. Out-of-Service Criteria", styles["SectionHead"]),
        Paragraph(
            "A vehicle is pulled out of service immediately, without waiting for the next scheduled interval, if any "
            "of the following are found: tire tread below the legal minimum, a brake warning that does not clear on "
            "restart, a cracked windshield that obstructs the driver's view, or a fluid leak that is active rather "
            "than a residual stain.",
            styles["Body"],
        ),
        Paragraph("5. Roadside Breakdown Procedure", styles["SectionHead"]),
        Paragraph(
            "If a vehicle breaks down on the road, the driver places the hazard triangle behind the vehicle, turns "
            "on hazard lights, and calls the roadside assistance line before doing anything else. Drivers do not "
            "attempt repairs on an active roadway; the vehicle is towed to the nearest depot or an approved partner "
            "garage.",
            styles["Body"],
        ),
        Paragraph("6. Maintenance Logging Requirement", styles["SectionHead"]),
        Paragraph(
            "Every service event, scheduled or roadside, is logged in the fleet maintenance system within 24 hours, "
            "including odometer reading, parts replaced, and the depot mechanic who performed the work.",
            styles["Body"],
        ),
    ]
    doc5.build(story)
    print(f"  wrote {PDF_DIR / 'vehicle_maintenance_manual.pdf'}")

    print("Done. Replace any of these with your own PDFs anytime by dropping them into", PDF_DIR.resolve())

In [ ]:
from typing import List
from datetime import datetime

from langchain_core.documents import Document

pdf_files = sorted(PDF_DIR.glob('*.pdf'))

print('PDF folder exists:', PDF_DIR.exists())
print('PDF count:', len(pdf_files))
for p in pdf_files:
    print(' -', p.name)

## 3) Load PDFs

This helper loads each PDF into a list of `Document` objects and attaches basic metadata such as file name and page number.

In [ ]:
from pypdf import PdfReader

def load_pdf(path: Path) -> List[Document]:
    reader = PdfReader(str(path))
    docs: List[Document] = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        if text.strip():
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        'source': str(path),
                        'file_name': path.name,
                        'type': 'pdf',
                        'page': page_num,
                        'ingested_at': datetime.utcnow().isoformat(),
                    },
                )
            )
    return docs

all_docs: List[Document] = []
for pdf in pdf_files:
    all_docs.extend(load_pdf(pdf))

print('Loaded documents:', len(all_docs))
if all_docs:
    print('Example metadata:', all_docs[0].metadata)

## 4) Add a small metadata helper

Metadata helps later when you filter or inspect retrieved chunks.

A simple metadata set is enough for a basic lab:

- `source`
- `file_name`
- `page`
- `type`
- `chunk_config`

In [ ]:
def add_chunk_metadata(docs: List[Document], chunk_config: str) -> List[Document]:
    enriched = []
    for doc in docs:
        meta = dict(doc.metadata)
        meta['chunk_config'] = chunk_config
        enriched.append(Document(page_content=doc.page_content, metadata=meta))
    return enriched

## 5) Define 3 chunking configurations

To keep the lab basic, we will use three different character-based chunking setups:

- small chunks
- medium chunks
- large chunks

The LangChain docs explain that splitters break large documents into smaller chunks so they fit retrieval and context-window limits.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_configs = {
    'small': RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50),
    'medium': RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100),
    'large': RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150),
}

chunk_configs

## 6) Split documents with each configuration

We will generate three chunk sets and compare their sizes.

In [ ]:
chunk_sets = {}
chunk_stats = []

for name, splitter in chunk_configs.items():
    split_docs = splitter.split_documents(all_docs)
    split_docs = add_chunk_metadata(split_docs, name)
    chunk_sets[name] = split_docs
    chunk_stats.append((name, len(split_docs)))

chunk_stats

In [ ]:
for name, docs in chunk_sets.items():
    print(f'\n=== {name.upper()} ===')
    print('Chunks:', len(docs))
    if docs:
        print('Sample metadata:', docs[0].metadata)
        print('Sample text preview:', docs[0].page_content[:300].replace('\n', ' '))

## 7) Create a local CPU embedding model

LangChain's Hugging Face docs say `HuggingFaceEmbeddings` can run open-source embedding models locally. The embeddings docs also note that local CPU models can be suitable for short queries and that hosted APIs introduce network latency.

This cell tries the real model first, and falls back to a deterministic offline hashing embedding if it can't be downloaded -- e.g. no internet access to Hugging Face. The offline fallback is noticeably rougher than true semantic embeddings, but it keeps the rest of the lab runnable.

In [ ]:
import hashlib
from langchain_core.embeddings import Embeddings


class OfflineHashEmbeddings(Embeddings):
    """Deterministic, dependency-free embeddings for sandboxes without network
    access to Hugging Face. Uses a fixed-dimension hashing trick so Chroma can
    still perform real similarity search based on shared vocabulary."""

    def __init__(self, dim: int = 384):
        self.dim = dim

    def _embed(self, text: str):
        vec = [0.0] * self.dim
        for token in text.lower().split():
            h = int(hashlib.sha256(token.encode("utf-8")).hexdigest(), 16)
            vec[h % self.dim] += 1.0
        norm = sum(v * v for v in vec) ** 0.5
        if norm > 0:
            vec = [v / norm for v in vec]
        return vec

    def embed_documents(self, texts):
        return [self._embed(t) for t in texts]

    def embed_query(self, text):
        return self._embed(text)


try:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    embeddings.embed_query("connectivity check")
    print('Embedding model ready on CPU (HuggingFace all-MiniLM-L6-v2, live).')
except Exception as e:
    embeddings = OfflineHashEmbeddings(dim=384)
    print(f'Embedding model ready: offline hashing fallback ({type(e).__name__}).')

## 8) Store each chunk set in Chroma

We will save each chunk configuration into a separate local collection directory.

This keeps the results easy to compare and inspect.

In [ ]:
from langchain_chroma import Chroma

base_dir = Path('./chroma_labs')
base_dir.mkdir(exist_ok=True)

stores = {}

for name, docs in chunk_sets.items():
    persist_dir = base_dir / name
    store = Chroma(
        collection_name=f'pdf_lab_{name}',
        embedding_function=embeddings,
        persist_directory=str(persist_dir),
    )
    store.add_documents(docs)
    stores[name] = store
    print(f'Saved {len(docs)} chunks to {persist_dir}')

## 9) Compare retrieval from each configuration

We can now test how each chunk size behaves with a sample question.

In [ ]:
query = 'What are the hours of service and rest break rules for drivers?'

for name, store in stores.items():
    print(f'\n--- {name.upper()} ---')
    results = store.similarity_search(query, k=2)
    for i, doc in enumerate(results, 1):
        print(f'Result {i}:', doc.metadata.get('file_name'), 'page', doc.metadata.get('page'))
        print(doc.page_content[:250].replace('\n', ' '))
        print()

## 10) Inspect embedding latency

The LangChain embeddings docs explain that local CPU models reduce network round trips but still take time to run. This cell times a small batch so you can get a rough feel for local embedding cost.

In [ ]:
import time

sample_texts = [doc.page_content for doc in all_docs[:5]]

start = time.perf_counter()
vectors = embeddings.embed_documents(sample_texts)
elapsed = time.perf_counter() - start

print('Embedded texts:', len(vectors))
print(f'Elapsed seconds: {elapsed:.3f}')
print('Vector dimension:', len(vectors[0]) if vectors else 0)

## 11) Basic summary table

This helps compare the three chunking setups at a glance.

In [ ]:
summary = []
for name, docs in chunk_sets.items():
    summary.append({
        'config': name,
        'chunks': len(docs),
        'persist_dir': str(base_dir / name),
    })

summary

## 12) Clean-up helper

Use this when you want to remove the local vector-store directories and start over.

In [ ]:
import shutil

def cleanup_lab():
    if base_dir.exists():
        shutil.rmtree(base_dir)
        print('Removed', base_dir)
    else:
        print('Nothing to clean')

# cleanup_lab()

## 13) What to try next

Once this lab works end-to-end, you can extend it later with:

- retriever queries scoped to a single document, e.g. only the Hazmat Handling Procedures
- metadata filtering, e.g. only `chunk_config == 'medium'`
- reranking
- evaluation
- LangSmith retriever traces

LangChain's knowledge-base docs describe loading, splitting, embedding, and storing as the core retrieval building blocks.

## Key takeaways

- PDFs are loaded into standardized `Document` objects.
- Metadata should be attached during ingestion.
- Chunk size changes retrieval behavior.
- Local Hugging Face embeddings can run on CPU, with a deterministic offline fallback when there's no network access.
- Chroma can persist vector data locally.
- Comparing multiple chunking setups is a good first lab before moving into deeper RAG work over Portside's real policy documents.

## Verification status in this environment

| Component | Status |
|---|---|
| Sample PDF generation (5 Portside Logistics documents) | Executed and verified |
| PDF loading, 3 chunking configurations, chunk-count comparison | Executed and verified (38 / 20 / 13 chunks for small / medium / large) |
| Chroma storage and cross-config retrieval comparison | Executed and verified, using the offline hashing embeddings fallback |
| `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) | Not verified here -- this sandbox has no network route to `huggingface.co`. Falls back automatically; swap back in on a machine with internet access for genuinely semantic retrieval. |

## References

- PDF loaders: https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs
- Seeding the vector store: https://docs.langchain.com/oss/python/langchain/knowledge-base#seeding-the-vector-store
- Splitting documents: https://docs.langchain.com/oss/python/langchain/rag#splitting-documents
- Hugging Face embeddings: https://docs.langchain.com/oss/python/integrations/providers/huggingface#huggingfaceembeddings
- Embeddings latency: https://docs.langchain.com/oss/python/integrations/embeddings/index#latency